# Metadata per study

Input: [syn77563814](https://www.synapse.org/Synapse:syn77563814), the metadata index of the 7 studies.

Output: 6 CSVs in `data/metadata/by_study/`, one row per sequencing run (`SRR`).


**6 studies** each describe their samples with **3 metadata files**, nested one inside the next:

```
subject      (1 row = 1 person: sex, age, diagnosis)
   |  linked by globalSubjectId
biospecimen  (1 row = 1 tissue sample: brain region, cell type, PMI)
   |  linked by GSM
assay        (1 row = 1 sequencing run: SRR, RIN, library prep)
                SRR = a column in the count matrix
```

**1 study** (`syn68932658`, HEK293T) is a cell line, contains sra_run_table_synid only, not included

## 1. Setup

In [4]:
import logging
from pathlib import Path

import pandas as pd
import synapseclient
from IPython.display import display
from synapseclient.operations import get

OUT_DIR = Path.cwd().parent.parent / "data" / "metadata" / "by_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.getLogger("synapseclient_default").setLevel(logging.ERROR)   # quiet download chatter

synapseclient.login(silent=True)

# the index: which 3 metadata files belong to which study
metadata = pd.read_csv(get("syn77563814").path)

metadata

,dataset_name,dataset_synid,dataset_folder_synid,metadata_folder_synid,assay_metadata_synid,biospecimen_metadata_synid,subject_metadata_synid,sra_run_table_synid,notes
0,Unexpected similarities between C9ORF72 and sp...,syn67729513,syn68800131,syn69041268,syn69041363,syn69041360,syn69041358,NaN,NaN
1,Neurovascular dysfunction in GRN-associated fr...,syn67719824,syn68800127,syn69041265,syn69041361,syn69041357,syn69041355,NaN,NaN
2,Loss of Nuclear TDP-43 Is Associated with Deco...,syn67754663,syn69949949,syn69950044,syn69950045,syn69950047,syn69950046,NaN,Dataset is under parent folder syn69041387; me...
3,Postmortem Cortex Samples Identify Distinct Mo...,syn67737779,syn68800140,syn69041266,syn69041356,syn69041359,syn69041364,NaN,NaN
4,Divergent impacts of C9orf72 repeat expansion ...,syn67743547,syn68916799,syn69041269,syn69041381,syn69041377,syn69041378,NaN,Dataset is under SuperSeries parent folder syn...
5,RNA methylation influences TDP43 binding and d...,syn68932658,syn68753718,syn69009342,NaN,NaN,NaN,syn68917187,Located under SRA Datasets folder syn68922789
6,Truncated stathmin-2 is a marker of TDP-43 pat...,syn67733559,syn68800135,syn69041267,syn69041365,syn69041362,syn69041354,NaN,NaN


In [6]:
metadata = metadata[["dataset_synid", 
                     "assay_metadata_synid", "biospecimen_metadata_synid", "subject_metadata_synid",
                     "sra_run_table_synid"]]
metadata

,dataset_synid,assay_metadata_synid,biospecimen_metadata_synid,subject_metadata_synid,sra_run_table_synid
0,syn67729513,syn69041363,syn69041360,syn69041358,NaN
1,syn67719824,syn69041361,syn69041357,syn69041355,NaN
2,syn67754663,syn69950045,syn69950047,syn69950046,NaN
3,syn67737779,syn69041356,syn69041359,syn69041364,NaN
4,syn67743547,syn69041381,syn69041377,syn69041378,NaN
5,syn68932658,NaN,NaN,NaN,syn68917187
6,syn67733559,syn69041365,syn69041362,syn69041354,NaN



The metadata [syn77563814](https://www.synapse.org/Synapse:syn77563814) is ONLY a lookup table with synid. It points to each study’s:
- 6 studies have 3 GEO-derived files each (assay, biospecimen, subject).
- 1 study has is derived from cell line only (HEK293T) and has SraRunTable only

## 2. Process the 6 studies with the metadata

Process 6 studies to join the linked 3 metadata checks

In [74]:
STUDIES = ["syn67729513", "syn67719824", "syn67754663",
           "syn67737779", "syn67743547", "syn67733559"]

for study_id in STUDIES:
    print("=" * 70, "\n", study_id, "\n", "=" * 70)
    row = metadata[metadata["dataset_synid"] == study_id].iloc[0]

    # Load assay, bio, subject
    assay = pd.read_csv(get(row["assay_metadata_synid"]).path)
    bio = pd.read_csv(get(row["biospecimen_metadata_synid"]).path)
    subject = pd.read_csv(get(row["subject_metadata_synid"]).path)

    print(f"assay        {len(assay):5d} rows   one per sequencing run")
    print(f"biospecimen  {len(bio):5d} rows   one per tissue sample")
    print(f"subject      {len(subject):5d} rows   one per person")
    # Hide the clinical info
    display(assay.columns)
    display(bio.columns)
    display(subject.columns)

    print("runs whose GSM is missing from biospecimen:", len(set(assay["GSM"]) - set(bio["GSM"])))
    print("runs per GSM:", assay.groupby("GSM")["SRR"].count().value_counts().to_dict())

    # run -> tissue sample
    df = assay.merge(bio[["GSM"] + [c for c in bio.columns if c not in assay.columns]],
                     on="GSM", how="left")

    # tissue sample -> person  (this is the hop that brings in sex)
    df = df.merge(subject[["globalSubjectId"] + [c for c in subject.columns if c not in df.columns]],
                  on="globalSubjectId", how="left")

    # sort by run, and put SRR first
    df = df.sort_values("SRR")
    df = df[["SRR"] + [c for c in df.columns if c != "SRR"]]

    assert len(df) == len(assay), "the join changed the row count"
    assert df["SRR"].is_unique, "duplicate SRR"

    # Hide the clinical info
    display(df.columns)

    out = OUT_DIR / f"metadata_{study_id}_by_run.csv"
    df.to_csv(out, index=False)
    print(f"wrote {out.name}: {len(df)} runs x {df.shape[1]} columns\n")

 syn67729513 
assay           77 rows   one per sequencing run
biospecimen     77 rows   one per tissue sample
subject         75 rows   one per person


Index(['GEOSuperSeries', 'GSE', 'GSM', 'SRX', 'SRS', 'SRR', 'BioProject', 'BioSample', 'globalSubjectId', 'originalSubjectId', 'originalSampleName', 'readLength', 'totalReads', 'libraryLayout',
       'platform', 'assay', 'dataType', 'dataSubtype', 'nucleicAcidSource', 'genomeReference', 'libraryPreparationMethod', 'softwareAndVersion', 'RIN'],
      dtype='str')

Index(['GEOSuperSeries', 'GSE', 'GSM', 'SRX', 'SRS', 'SRR', 'BioProject', 'BioSample', 'originalSampleName', 'species', 'originalSubjectId', 'biospecimenType', 'CNSRegion', 'isPostMortem', 'PMI',
       'globalSubjectId'],
      dtype='str')

Index(['GEOSuperSeries', 'GSE', 'globalSubjectId', 'originalSubjectId', 'species', 'disease', 'sex', 'age', 'race', 'comorbidities', 'diseaseDurationYears'], dtype='str')

runs whose GSM is missing from biospecimen: 0
runs per GSM: {1: 77}


...

Index(['GEOSuperSeries', 'GSE', 'GSM', 'SRX', 'SRS', 'SRR', 'BioProject', 'BioSample', 'globalSubjectId', 'originalSubjectId', 'originalSampleName', 'species', 'biospecimenType', 'CNSRegion',
       'BrodmannArea', 'isPostMortem', 'PMI'],
      dtype='str')

Index(['GEOSuperSeries', 'GSE', 'globalSubjectId', 'originalSubjectId', 'species', 'disease', 'sex', 'age', 'race', 'comorbidities', 'diseaseDurationYears'], dtype='str')

runs whose GSM is missing from biospecimen: 0
runs per GSM: {1: 803, 2: 196}


Index(['SRR', 'GEOSuperSeries', 'GSE', 'GSM', 'SRX', 'SRS', 'BioProject', 'readLength', 'totalReads', 'libraryLayout', 'platform', 'originalSampleName', 'assay', 'dataType', 'dataSubtype',
       'nucleicAcidSource', 'BioSample', 'genomeReference', 'libraryPreparationMethod', 'softwareAndVersion', 'originalSubjectId', 'globalSubjectId', 'RIN', 'species', 'biospecimenType', 'CNSRegion',
       'BrodmannArea', 'isPostMortem', 'PMI', 'disease', 'sex', 'age', 'race', 'comorbidities', 'diseaseDurationYears'],
      dtype='str')

wrote metadata_syn67733559_by_run.csv: 1195 runs x 35 columns



## Check the HEK293T (syn68932658)

The syn68932658 contains only cell line, and contains `sra_run_table_synid` only, no donar information 
(`age`, `disease`, `race`, `PMI`, `comorbidities`) are available, so no sensitive data

In [77]:
row = metadata[metadata["dataset_synid"] == "syn68932658"].iloc[0]
df = pd.read_csv(get(row["sra_run_table_synid"]).path)
df.columns

Index(['Run', 'AGE', 'Assay Type', 'AvgSpotLen', 'Bases', 'BIOMATERIAL_PROVIDER', 'BioProject', 'BioSample', 'BioSampleModel', 'Bytes', 'cell_line', 'Center Name', 'Consent', 'DATASTORE filetype',
       'DATASTORE provider', 'DATASTORE region', 'Experiment', 'Instrument', 'isolate', 'Library Name', 'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism', 'Platform', 'ReleaseDate',
       'replicate (run)', 'replicate', 'create_date', 'version', 'Sample Name', 'sex', 'SRA Study', 'tissue', 'treatment'],
      dtype='str')

## Upload the derived metadata for the 6 files

Upload the derived metadata

- `used` -- the index plus **that study's** three metadata files, so the graph shows
  `metadata_syn67729513_by_run.csv` came from `syn67729513`'s assay / biospecimen / subject
  files and nothing else
- `executed` -- this notebook (need to update the link later)

In [ ]:
from synapseclient.models import Activity, File, UsedEntity, UsedURL
from synapseclient.operations import store

PARENT = "syn77563799"
NOTEBOOK = None          # Use the URL of the notebook if available

for f in sorted(OUT_DIR.glob("metadata_syn*_by_run.csv")):
    study_id = f.name.replace("metadata_", "").replace("_by_run.csv", "")
    row = metadata[metadata["dataset_synid"] == study_id].iloc[0]

    # link the data
    sources = ["syn77563814", study_id,
               row["assay_metadata_synid"],
               row["biospecimen_metadata_synid"],
               row["subject_metadata_synid"]]
    sources = [s for s in sources if isinstance(s, str) and s.startswith("syn")]

    if NOTEBOOK is None:
        executed = []
    else:
        executed = [UsedURL(name="extract_metadata_per_study.ipynb", url=NOTEBOOK)]

    uploaded = store(File(
        path=str(f),
        parent_id=PARENT,
        description=f"One row per sequencing run for {study_id}: assay joined to biospecimen "
                    f"on GSM, then to subject on globalSubjectId.",
        activity=Activity(
            name=f"harmonize {study_id} metadata to run level",
            description="Two left joins at run.",
            used=[UsedEntity(target_id=s) for s in sources],
            executed=executed,
        ),
    ))
    print(f"{f.name:44s} -> {uploaded.id}")